In [ ]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability

In [ ]:
Logger().init_logger(None, None, logging_level="WARNING")
animal_ids = [6]
paradigm = [1100]
session_range = [0,33]
session_ids = None
normalize = True
smooth = False
load_fr_track = False
cast_numeric_float32 = True
excl_session_names = ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

In [ ]:
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
if cast_numeric_float32:
    num_cols = fr.select_dtypes(include=[np.number]).columns
    fr[num_cols] = fr[num_cols].astype(np.float32)

fr_track = None
if load_fr_track:
    fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
    if cast_numeric_float32:
        num_cols = fr_track.select_dtypes(include=[np.number]).columns
        fr_track[num_cols] = fr_track[num_cols].astype(np.float32)

fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names) # z-scoring within sessions
if cast_numeric_float32:
    num_cols = fr_z_scored.select_dtypes(include=[np.number]).columns
    fr_z_scored[num_cols] = fr_z_scored[num_cols].astype(np.float32)

ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)
if cast_numeric_float32:
    num_cols = ensamble_proj.select_dtypes(include=[np.number]).columns
    ensamble_proj[num_cols] = ensamble_proj[num_cols].astype(np.float32)
behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

In [ ]:
counts = fr["Unit0002"].value_counts()
percentages = fr["Unit0002"].value_counts(normalize=True) * 100

result = pd.DataFrame({
    "count": counts,
    "percentage": percentages
})
result

In [ ]:
session_counts = (
    fr.groupby(level="session_id")
    .size()
    .rename("n_entries")
    .reset_index()
)
# convert number of bins → minutes
session_counts["duration_min"] = session_counts["n_entries"] * 0.04 / 60
session_counts["session_date"] = session_counts["session_id"].str.split("_").str[0]
session_counts = session_counts.sort_values("session_id")
duplicate_session_dates = session_counts["session_date"].duplicated(keep=False)
session_date_number = session_counts.groupby("session_date").cumcount() + 1
session_counts["session_plot_label"] = np.where(
    duplicate_session_dates,
    session_counts["session_date"] + " (" + session_date_number.astype(str) + ")",
    session_counts["session_date"],
)
long_session_color = "#215CAF"
other_session_color = "#B7352D"
session_counts["duration_color"] = np.where(
    session_counts["duration_min"] > 10,
    long_session_color,
    other_session_color,
)

fig = px.bar(
    session_counts,
    x="session_plot_label",
    y="duration_min",
    labels={
        "session_plot_label": "Session",
        "duration_min": "Recording length (min)",
    },
    title="Recording length per session",
)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=session_counts["session_plot_label"],
    showgrid=False,
    title_standoff=4,
    tickangle=-45,
    tickfont=dict(size=7, family="Arial"),
    title_font=dict(size=8, family="Arial"),
)

y_tick_step = 10
y_tick_max = max(10, int(np.ceil(session_counts["duration_min"].max() / y_tick_step) * y_tick_step))
y_tickvals = np.arange(0, y_tick_max + y_tick_step, y_tick_step)
y_ticktext = [
    "<span style='color:#B7352D'>10</span>" if tick == 10 else f"{tick:g}"
    for tick in y_tickvals
]

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(128,128,128,0.35)",
    gridwidth=0.5,
    tickmode="array",
    tickvals=y_tickvals,
    ticktext=y_ticktext,
    tickfont=dict(size=7, family="Arial"),
    title_font=dict(size=8, family="Arial"),
)

fig.update_traces(marker_color=session_counts["duration_color"].tolist())

fig.add_hline(y=10, line_dash="dash", line_color="#B7352D", line_width=1)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(text="Recording length per session", font=dict(size=8, family="Arial")),
    font=dict(family="Arial", size=7),
    width=400,
    height=270,
    margin=dict(l=50, r=20, t=30, b=90),
)

fig.show()
fig.write_image(
    "animal_6_rec_length.svg",
    width=fig.layout.width,
    height=fig.layout.height,
    scale=1,
)

In [ ]:
behav.index.get_level_values('session_id').unique()

In [ ]:
res_cue1 = []
res_cue2 = []
d_stop_c1 = []
d_stop_c2 = []
r1_stop = []
r2_stop = []

r1_wrong_only = []
r2_wrong_only = []

for s_id in behav.index.unique("session_id"):
    s_behav = behav.loc[s_id]

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["cue"] == 1)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 1]) * 100
    res_cue1.append((s_id, prop_expert_trials))

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1) &
        (s_behav["cue"] == 2)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 2]) * 100
    res_cue2.append((s_id, prop_expert_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 1]) * 100
    d_stop_c1.append((s_id, prop_d_stop_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 2]) * 100
    d_stop_c2.append((s_id, prop_d_stop_trials))

    r1_wrong = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1)
    ]
    r2_wrong = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["choice_R1"] == 1)
    ]

    r1_wrong_only.append((s_id, len(r1_wrong) / len(s_behav) * 100))
    r2_wrong_only.append((s_id, len(r2_wrong) / len(s_behav) * 100))

    r1_stops = s_behav[s_behav["choice_R1"] == 1]
    r2_stops = s_behav[s_behav["choice_R2"] == 1]

    prop_r1_stop = len(r1_stops) / len(s_behav) * 100
    prop_r2_stop = len(r2_stops) / len(s_behav) * 100
    r1_stop.append((s_id, prop_r1_stop))
    r2_stop.append((s_id, prop_r2_stop))

# numpy conversions
d_stop_c1 = np.array(d_stop_c1, dtype=object)
d_stop_c2 = np.array(d_stop_c2, dtype=object)
res_cue1  = np.array(res_cue1,  dtype=object)
res_cue2  = np.array(res_cue2,  dtype=object)
r1_wrong_only = np.array(r1_wrong_only, dtype=object)
r2_wrong_only = np.array(r2_wrong_only, dtype=object)

idx1 = np.argsort(res_cue1[:, 0])
idx2 = np.argsort(res_cue2[:, 0])

x1, y1 = res_cue1[idx1, 0], res_cue1[idx1, 1].astype(float)
x2, y2 = res_cue2[idx2, 0], res_cue2[idx2, 1].astype(float)
x3, y3 = d_stop_c1[idx1, 0], d_stop_c1[idx1, 1].astype(float)
x4, y4 = d_stop_c2[idx2, 0], d_stop_c2[idx2, 1].astype(float)

x5y5 = np.array(r1_stop, dtype=object)
x5, y5 = x5y5[:, 0], x5y5[:, 1].astype(float)

x6y6 = np.array(r2_stop, dtype=object)
x6, y6 = x6y6[:, 0], x6y6[:, 1].astype(float)

x7, y7 = r1_wrong_only[:, 0], r1_wrong_only[:, 1].astype(float)
x8, y8 = r2_wrong_only[:, 0], r2_wrong_only[:, 1].astype(float)

# date-based split (STRING COMPARISON)
x_pre_cue = "2024-11-27_16-11"

m5 = x5 <= x_pre_cue
x5, y5 = x5[m5], y5[m5]

m6 = x6 <= x_pre_cue
x6, y6 = x6[m6], y6[m6]

m1 = x1 >= x_pre_cue
x1, y1 = x1[m1], y1[m1]

m2 = x2 >= x_pre_cue
x2, y2 = x2[m2], y2[m2]

m3 = x3 >= x_pre_cue
x3, y3 = x3[m3], y3[m3]

m4 = x4 >= x_pre_cue
x4, y4 = x4[m4], y4[m4]

# average expert performance (Cue 1 and Cue 2) per session
cue1_expert = pd.Series(res_cue1[:, 1].astype(float), index=res_cue1[:, 0], name="cue1_expert")
cue2_expert = pd.Series(res_cue2[:, 1].astype(float), index=res_cue2[:, 0], name="cue2_expert")
avg_expert = pd.concat([cue1_expert, cue2_expert], axis=1).mean(axis=1).sort_index()
x_avg_expert = avg_expert.index.to_numpy()
y_avg_expert = avg_expert.to_numpy(dtype=float)
m_avg = x_avg_expert >= x_pre_cue
x_avg_expert, y_avg_expert = x_avg_expert[m_avg], y_avg_expert[m_avg]

# mean Assembly023 activation per session
if "session_id" in ensamble_proj.index.names:
    assembly023_session_mean = ensamble_proj.groupby(level="session_id")["Assembly023"].mean()
elif "session_id" in ensamble_proj.columns:
    assembly023_session_mean = ensamble_proj.groupby("session_id")["Assembly023"].mean()
else:
    raise KeyError("Could not find 'session_id' in ensamble_proj index names or columns")

if "session_id" in behav.index.names:
    behav_sessions = pd.Index(behav.index.get_level_values("session_id").unique()).astype(str)
else:
    behav_sessions = pd.Index(behav.index.unique()).astype(str)

assembly023_session_mean.index = assembly023_session_mean.index.astype(str)
common_sessions = np.intersect1d(behav_sessions.to_numpy(), assembly023_session_mean.index.to_numpy())
assembly023_session_mean = assembly023_session_mean.loc[common_sessions].sort_index()
x_ens023 = assembly023_session_mean.index.to_numpy()
y_ens023 = assembly023_session_mean.to_numpy(dtype=float)

# plotting
fig = go.Figure()

fig.add_trace(go.Scatter(x=x5, y=y5, mode="lines+markers",
    name="R1 Stop Overall",
    line=dict(width=2, color="lightgrey"),
    marker=dict(size=7, color="lightgrey"),
))

fig.add_trace(go.Scatter(x=x6, y=y6, mode="lines+markers",
    name="R2 Stop Overall",
    line=dict(width=2, color="dimgrey"),
    marker=dict(size=7, color="dimgrey"),
))

fig.add_trace(go.Scatter(x=x1, y=y1, mode="lines+markers",
    name="Cue 1 - Expert",
    line=dict(width=2, color="orange"),
    marker=dict(size=7, color="orange"),
))

fig.add_trace(go.Scatter(x=x3, y=y3, mode="lines+markers",
    name="Cue 1 - Double Stop",
    line=dict(width=2, color="peachpuff", dash="dash"),
    marker=dict(size=7, color="peachpuff"),
))

fig.add_trace(go.Scatter(x=x2, y=y2, mode="lines+markers",
    name="Cue 2 - Expert",
    line=dict(width=2, color="purple"),
    marker=dict(size=7, color="purple"),
))

fig.add_trace(go.Scatter(x=x4, y=y4, mode="lines+markers",
    name="Cue 2 - Double Stop",
    line=dict(width=2, color="lavender", dash="dash"),
    marker=dict(size=7, color="lavender"),
))

fig.add_trace(go.Scatter(x=x_avg_expert, y=y_avg_expert, mode="lines+markers",
    name="Average Expert",
    line=dict(width=3, color="red", dash="dot"),
    marker=dict(size=7, color="red"),
))

fig.add_trace(go.Scatter(x=x_ens023, y=y_ens023, mode="lines+markers",
    name="Assembly023 Mean Activation",
    line=dict(width=2, color="blue"),
    marker=dict(size=6, color="blue"),
    yaxis="y2",
))

# fig.add_trace(go.Scatter(
#     x=x7, y=y7,
#     mode="lines+markers",
#     name="C1-Wrong only(R2)",
#     line=dict(width=2, color="lightcoral"),
#     marker=dict(size=7, color="lightcoral"),
# ))

# fig.add_trace(go.Scatter(
#     x=x8, y=y8,
#     mode="lines+markers",
#     name="C2-Wrong only(R1)",
#     line=dict(width=2, color="indianred"),
#     marker=dict(size=7, color="indianred"),
# ))


# fig.add_trace(go.Scatter(
#     x=x2, y=m2 * x2_num + b2,
#     mode="lines",
#     name="Cue 2 fit",
#     line=dict(width=2, color="purple", dash="dash"),
# ))

all_x_ticks = np.sort(np.unique(np.concatenate([x5, x6, x1, x2, x3, x4, x_avg_expert, x_ens023])))
all_x_ticklabels = [s.split("_")[0] for s in all_x_ticks]

# layout
fig.update_layout(
    title=dict(text="Animal Performance (Expert Trials and Double Stops)", font=dict(size=10, family="Arial")),
    width=900,
    height=340,
    margin=dict(l=55, r=70, t=35, b=95),
    xaxis_title="Session",
    yaxis_title="Proportion of trials (%)",
    yaxis2=dict(
        title="Assembly023 Mean Activation",
        overlaying="y",
        side="right",
        showgrid=False,
        zeroline=False,
        showline=True,
        title_font=dict(size=10, family="Arial"),
        tickfont=dict(size=9, family="Arial"),
    ),
    plot_bgcolor="white",
    font=dict(family="Arial", size=9),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="white",
        bordercolor="rgba(0,0,0,0.2)",
        borderwidth=1,
        xanchor="left", yanchor="top",
        font=dict(size=9, family="Arial"),
        tracegroupgap=0,
    ),
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.2)",
    zeroline=False,
    showline=True,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)

fig.update_xaxes(
    tickmode="array",
    tickvals=all_x_ticks,
    ticktext=all_x_ticklabels,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.1)",
    showline=True,
    zeroline=False,
    tickangle=-45,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)
fig.update_traces(marker=dict(size=6))

fig.show()


In [ ]:
# ensamble_proj already loaded above; avoid re-loading to keep memory stable
ensamble_proj.head()

In [ ]:
# Firing Rate stability grand average -> increasing fr over time? 
fr_units = fr.loc[:, fr.columns.str.startswith("Unit")]
session_mean_hz = fr_units.groupby("session_id").mean()

session_mean_z = (session_mean_hz - session_mean_hz.mean(axis=0)) / session_mean_hz.std(axis=0)

pop_mean_z = session_mean_z.mean(axis=1)

fig = px.line(pop_mean_z, title="Session drift (z-scored session means)")
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()


In [ ]:
fr_units = fr_z_scored.loc[:, fr.columns.str.startswith("Unit")]

Z_unit_session = fr_units.groupby("session_id").mean().T  # Units x Sessions

fig = px.imshow(
    Z_unit_session,
    aspect="auto",
    labels=dict(x="Session", y="Unit", color="Mean z"),
    title="Per-neuron drift: session-mean z-score"
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()


In [ ]:
if fr_track is None:
    print("fr_track not loaded (load_fr_track=False). Set it to True in the config cell when needed.")
else:
    print(f"fr_track shape: {fr_track.shape}")
    fr_track.head()

In [ ]:
# Animal performance plot copy without Assembly012 overlay; exclude short sessions (<10 trials)
min_trials_per_session = 10

trial_counts = (
    behav.reset_index()[["session_id", "trial_id"]]
    .dropna()
    .drop_duplicates()
    .groupby("session_id")["trial_id"]
    .nunique()
    .sort_index()
)
valid_sessions = pd.Index(trial_counts[trial_counts >= min_trials_per_session].index.astype(str))
excluded_short_sessions = trial_counts[trial_counts < min_trials_per_session].index.astype(str).tolist()

print(f"Excluded sessions with <{min_trials_per_session} trials: {excluded_short_sessions}")

behav_perf = behav.loc[behav.index.get_level_values("session_id").isin(valid_sessions)].copy()

res_cue1 = []
res_cue2 = []
d_stop_c1 = []
d_stop_c2 = []
r1_stop = []
r2_stop = []

r1_wrong_only = []
r2_wrong_only = []

for s_id in behav_perf.index.unique("session_id"):
    s_behav = behav_perf.loc[s_id]

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["cue"] == 1)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 1]) * 100
    res_cue1.append((s_id, prop_expert_trials))

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1) &
        (s_behav["cue"] == 2)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 2]) * 100
    res_cue2.append((s_id, prop_expert_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 1]) * 100
    d_stop_c1.append((s_id, prop_d_stop_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 2]) * 100
    d_stop_c2.append((s_id, prop_d_stop_trials))

    r1_wrong = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1)
    ]
    r2_wrong = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["choice_R1"] == 1)
    ]

    r1_wrong_only.append((s_id, len(r1_wrong) / len(s_behav) * 100))
    r2_wrong_only.append((s_id, len(r2_wrong) / len(s_behav) * 100))

    r1_stops = s_behav[s_behav["choice_R1"] == 1]
    r2_stops = s_behav[s_behav["choice_R2"] == 1]

    prop_r1_stop = len(r1_stops) / len(s_behav) * 100
    prop_r2_stop = len(r2_stops) / len(s_behav) * 100
    r1_stop.append((s_id, prop_r1_stop))
    r2_stop.append((s_id, prop_r2_stop))

# numpy conversions
d_stop_c1 = np.array(d_stop_c1, dtype=object)
d_stop_c2 = np.array(d_stop_c2, dtype=object)
res_cue1 = np.array(res_cue1, dtype=object)
res_cue2 = np.array(res_cue2, dtype=object)
r1_wrong_only = np.array(r1_wrong_only, dtype=object)
r2_wrong_only = np.array(r2_wrong_only, dtype=object)

idx1 = np.argsort(res_cue1[:, 0])
idx2 = np.argsort(res_cue2[:, 0])

x1, y1 = res_cue1[idx1, 0], res_cue1[idx1, 1].astype(float)
x2, y2 = res_cue2[idx2, 0], res_cue2[idx2, 1].astype(float)
x3, y3 = d_stop_c1[idx1, 0], d_stop_c1[idx1, 1].astype(float)
x4, y4 = d_stop_c2[idx2, 0], d_stop_c2[idx2, 1].astype(float)

x5y5 = np.array(r1_stop, dtype=object)
x5, y5 = x5y5[:, 0], x5y5[:, 1].astype(float)

x6y6 = np.array(r2_stop, dtype=object)
x6, y6 = x6y6[:, 0], x6y6[:, 1].astype(float)

x7, y7 = r1_wrong_only[:, 0], r1_wrong_only[:, 1].astype(float)
x8, y8 = r2_wrong_only[:, 0], r2_wrong_only[:, 1].astype(float)

# date-based split (STRING COMPARISON)
x_pre_cue = "2024-11-27_16-11"

m5 = x5 <= x_pre_cue
x5, y5 = x5[m5], y5[m5]

m6 = x6 <= x_pre_cue
x6, y6 = x6[m6], y6[m6]

m1 = x1 >= x_pre_cue
x1, y1 = x1[m1], y1[m1]

m2 = x2 >= x_pre_cue
x2, y2 = x2[m2], y2[m2]

m3 = x3 >= x_pre_cue
x3, y3 = x3[m3], y3[m3]

m4 = x4 >= x_pre_cue
x4, y4 = x4[m4], y4[m4]

# average expert performance (Cue 1 and Cue 2) per session
cue1_expert = pd.Series(res_cue1[:, 1].astype(float), index=res_cue1[:, 0], name="cue1_expert")
cue2_expert = pd.Series(res_cue2[:, 1].astype(float), index=res_cue2[:, 0], name="cue2_expert")
avg_expert = pd.concat([cue1_expert, cue2_expert], axis=1).mean(axis=1).sort_index()
x_avg_expert = avg_expert.index.to_numpy()
y_avg_expert = avg_expert.to_numpy(dtype=float)
m_avg = x_avg_expert >= x_pre_cue
x_avg_expert, y_avg_expert = x_avg_expert[m_avg], y_avg_expert[m_avg]

# Recolour by the cue the animal actually SAW (cue_visible)
# The data/series are NOT changed. After the reversal the displayed cue and the
# rewarded zone both swap (1<->2), so the correct-zone "expert"/"double-stop" series
# for a given VISUAL cue is simply the other token series. We therefore keep every
# value and only swap which colour each series is drawn in for post-reversal sessions.
reversal_date = "2025-01-23"  # first post-reversal session (flip_Cue1R1_Cue2R2 True)

def _by_visual(x_pre_src, y_pre_src, x_post_src, y_post_src):
    pre_m = x_pre_src < reversal_date
    post_m = x_post_src >= reversal_date
    return (np.concatenate([x_pre_src[pre_m], x_post_src[post_m]]),
            np.concatenate([y_pre_src[pre_m], y_post_src[post_m]]))

# Visual cue 1 (orange / peachpuff): cue1 series before reversal, cue2 series after.
xv1_exp, yv1_exp = _by_visual(x1, y1, x2, y2)
xv1_ds,  yv1_ds  = _by_visual(x3, y3, x4, y4)
# Visual cue 2 (purple / lavender): cue2 series before reversal, cue1 series after.
xv2_exp, yv2_exp = _by_visual(x2, y2, x1, y1)
xv2_ds,  yv2_ds  = _by_visual(x4, y4, x3, y3)

# plotting
fig = go.Figure()

fig.add_trace(go.Scatter(x=x5, y=y5, mode="lines+markers",
    name="R1 Stop Overall",
    line=dict(width=2, color="lightgrey"),
    marker=dict(size=7, color="lightgrey"),
))

fig.add_trace(go.Scatter(x=x6, y=y6, mode="lines+markers",
    name="R2 Stop Overall",
    line=dict(width=2, color="dimgrey"),
    marker=dict(size=7, color="dimgrey"),
))

fig.add_trace(go.Scatter(x=xv1_exp, y=yv1_exp, mode="lines+markers",
    name="Cue 1 - Expert",
    line=dict(width=2, color="orange"),
    marker=dict(size=7, color="orange"),
))

fig.add_trace(go.Scatter(x=xv1_ds, y=yv1_ds, mode="lines+markers",
    name="Cue 1 - Double Stop",
    line=dict(width=2, color="peachpuff", dash="dash"),
    marker=dict(size=7, color="peachpuff"),
))

fig.add_trace(go.Scatter(x=xv2_exp, y=yv2_exp, mode="lines+markers",
    name="Cue 2 - Expert",
    line=dict(width=2, color="purple"),
    marker=dict(size=7, color="purple"),
))

fig.add_trace(go.Scatter(x=xv2_ds, y=yv2_ds, mode="lines+markers",
    name="Cue 2 - Double Stop",
    line=dict(width=2, color="lavender", dash="dash"),
    marker=dict(size=7, color="lavender"),
))

# fig.add_trace(go.Scatter(x=x_avg_expert, y=y_avg_expert, mode="lines+markers",
#     name="Average Expert",
#     line=dict(width=3, color="red", dash="dot"),
#     marker=dict(size=7, color="red"),
# ))

# Assembly012 overlay intentionally disabled in this copy.
# fig.add_trace(go.Scatter(x=x_ens012, y=y_ens012, mode="lines+markers",
#     name="Assembly012 Mean Activation",
#     line=dict(width=2, color="blue"),
#     marker=dict(size=6, color="blue"),
#     yaxis="y2",
# ))

# Put alternating legend entries into a nearby second column.
for trace_idx in (1, 3, 5):
    fig.data[trace_idx].legend = "legend2"

all_x_ticks = np.sort(np.unique(np.concatenate([x5, x6, x1, x2, x3, x4, x_avg_expert])))
all_x_tick_dates = pd.Series(all_x_ticks).str.split("_").str[0]
duplicate_x_tick_dates = all_x_tick_dates.duplicated(keep=False)
x_tick_date_number = all_x_tick_dates.groupby(all_x_tick_dates).cumcount() + 1
all_x_ticklabels = np.where(
    duplicate_x_tick_dates,
    all_x_tick_dates + " (" + x_tick_date_number.astype(str) + ")",
    all_x_tick_dates,
)

split_start_date = "2025-01-17"
split_end_date = "2025-01-23"
split_start_matches = [s for s in all_x_ticks if str(s).startswith(split_start_date)]
split_end_matches = [s for s in all_x_ticks if str(s).startswith(split_end_date)]

if not split_start_matches or not split_end_matches:
    raise ValueError(
        f"Could not place split line: expected plotted sessions starting with "
        f"{split_start_date!r} and {split_end_date!r}."
    )

split_start_session = split_start_matches[-1]
split_end_session = split_end_matches[0]
split_start_idx = np.where(all_x_ticks == split_start_session)[0][0]
split_end_idx = np.where(all_x_ticks == split_end_session)[0][0]
if split_start_idx >= split_end_idx:
    raise ValueError(
        f"Could not place split line: {split_start_session!r} is not before "
        f"{split_end_session!r}."
    )

split_line_x = (split_start_idx + split_end_idx) / 2

fig.add_shape(
    type="line",
    xref="x",
    yref="paper",
    x0=split_line_x,
    x1=split_line_x,
    y0=0,
    y1=1,
    line=dict(color="blue", width=2),
)

# layout
fig.update_layout(
    title=dict(text="Animal Performance (Expert Trials and Double Stops)", font=dict(size=10, family="Arial")),
    width=900,
    height=340,
    margin=dict(l=55, r=30, t=60, b=95),
    xaxis_title="Session",
    yaxis_title="Proportion of trials (%)",
    plot_bgcolor="white",
    font=dict(family="Arial", size=9),
    legend=dict(
        x=0.01, y=1.12,
        bgcolor="rgba(255,255,255,0)",
        bordercolor="rgba(0,0,0,0)",
        borderwidth=0,
        xanchor="left", yanchor="top",
        font=dict(size=9, family="Arial"),
        tracegroupgap=0,
    ),
    legend2=dict(
        x=0.16, y=1.12,
        bgcolor="rgba(255,255,255,0)",
        bordercolor="rgba(0,0,0,0)",
        borderwidth=0,
        xanchor="left", yanchor="top",
        font=dict(size=9, family="Arial"),
        tracegroupgap=0,
    ),
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.2)",
    zeroline=False,
    showline=True,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=all_x_ticks,
    tickmode="array",
    tickvals=all_x_ticks,
    ticktext=all_x_ticklabels,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.1)",
    showline=True,
    zeroline=False,
    tickangle=-45,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)
fig.update_traces(marker=dict(size=6))

fig.show()
fig.write_image(
    "animal_6_performance.svg",
    width=fig.layout.width,
    height=fig.layout.height,
    scale=1,
)


In [ ]:
# Sankey choice-flow plot for session 2024-12-10_17-20.
#
# Publication layout notes:
# - Cue -> R1 is split by the eventual R2 outcome, so the first stage already
#   shows the four downstream branches without changing node totals.
# - Static labels are limited to node names and percentage-only flow labels.
from collections import defaultdict

sankey_session = "2025-01-23_16-48"
sankey_cues_to_plot = (1, 2)

correct_color = "rgba(0, 128, 80, 0.82)"       # perfect strategy
light_green_color = "rgba(0, 158, 115, 0.42)"  # reaches reward, non-perfect path
other_color = "rgba(210, 0, 0, 0.78)"          # wrong path
double_skip_color = "rgba(210, 0, 0, 0.38)"    # double-skip wrong path
cue_colors = {1: "orange", 2: "purple"}
stop_node_color = "#5B6168"
skip_node_color = "#C7CCD1"

NODE_PAD_FRAC = 0.0
NODE_OVERLAP = 0.0
NODE_THICKNESS = 18
NODE_LABEL_FONT_SIZE = 9
FLOW_LABEL_FONT_SIZE = 10
PANEL_TITLE_FONT_SIZE = 14
FIG_WIDTH = 620
FIG_HEIGHT = 760
PANEL_DOMAINS = {
    1: dict(x=[0.11, 0.89], y=[0.58, 0.86]),
    2: dict(x=[0.11, 0.89], y=[0.13, 0.41]),
}
NODE_NAMES = {
    0: "Cue",
    1: "Stop R1",
    2: "Skip R1",
    3: "Stop R2",
    4: "Skip R2",
}


def prepare_choice_sankey_trials(behavior):
    required_cols = ["session_id", "trial_id", "cue", "choice_R1", "choice_R2"]
    trials = behavior.reset_index()[required_cols].copy()

    for col in ["trial_id", "cue", "choice_R1", "choice_R2"]:
        trials[col] = pd.to_numeric(trials[col], errors="coerce")

    trials = trials.dropna(subset=required_cols)
    trials = trials[trials["trial_id"] >= 0]
    trials = trials[trials["cue"].isin([1, 2])]
    trials["session_id"] = trials["session_id"].astype(str)

    trials = (
        trials
        .sort_values(["session_id", "trial_id"])
        .groupby(["session_id", "trial_id"], as_index=False, observed=True)
        .agg(cue=("cue", "first"), choice_R1=("choice_R1", "first"), choice_R2=("choice_R2", "first"))
    )
    return trials


def _fmt_pct(count, denominator):
    pct = 0 if denominator == 0 else count / denominator * 100
    return f"{pct:.2f}%"


def _paper_point(domain, local_x, local_y_from_top):
    x0, x1 = domain["x"]
    y0, y1 = domain["y"]
    return (
        x0 + local_x * (x1 - x0),
        y1 - local_y_from_top * (y1 - y0),
    )


def make_cue_sankey(trials, session_id, cue, domain):
    panel = trials[(trials["session_id"] == str(session_id)) & (trials["cue"] == cue)].copy()
    if panel.empty:
        raise ValueError(f"No trials found for session {session_id!r} and cue {cue}.")

    panel["stop_R1"] = panel["choice_R1"] > 0
    panel["stop_R2"] = panel["choice_R2"] > 0
    n_trials = len(panel)

    cue_node, stop_r1_node, skip_r1_node, stop_r2_node, skip_r2_node = 0, 1, 2, 3, 4
    node_colors = [cue_colors[cue], stop_node_color, skip_node_color, stop_node_color, skip_node_color]

    branch_counts = defaultdict(int)
    for row in panel.itertuples(index=False):
        r1_target = stop_r1_node if bool(row.stop_R1) else skip_r1_node
        r2_target = stop_r2_node if bool(row.stop_R2) else skip_r2_node
        branch_counts[(r1_target, r2_target)] += 1

    stop_r1_stop_r2 = branch_counts[(stop_r1_node, stop_r2_node)]
    stop_r1_skip_r2 = branch_counts[(stop_r1_node, skip_r2_node)]
    skip_r1_stop_r2 = branch_counts[(skip_r1_node, stop_r2_node)]
    skip_r1_skip_r2 = branch_counts[(skip_r1_node, skip_r2_node)]
    stop_r1_total = stop_r1_stop_r2 + stop_r1_skip_r2
    skip_r1_total = skip_r1_stop_r2 + skip_r1_skip_r2
    stop_r2_total = stop_r1_stop_r2 + skip_r1_stop_r2
    skip_r2_total = stop_r1_skip_r2 + skip_r1_skip_r2

    top_y = NODE_PAD_FRAC
    bot_y = 1.0 - NODE_PAD_FRAC
    span_h = bot_y - top_y

    node_x = [0.08, 0.50, 0.50, 0.92, 0.92]
    node_y = [
        top_y,
        top_y,
        top_y + stop_r1_total / n_trials * span_h - NODE_OVERLAP,
        top_y,
        top_y + stop_r2_total / n_trials * span_h - NODE_OVERLAP,
    ]
    node_h = [
        span_h,
        stop_r1_total / n_trials * span_h,
        skip_r1_total / n_trials * span_h,
        stop_r2_total / n_trials * span_h,
        skip_r2_total / n_trials * span_h,
    ]
    node_flow = [n_trials, stop_r1_total, skip_r1_total, stop_r2_total, skip_r2_total]
    node_plot_y = [top + height / 2 for top, height in zip(node_y, node_h)]

    ordered_links = [
        # First-stage links are split by their downstream R2 outcome so the
        # Cue -> R1 step already shows the four eventual branches.
        (cue_node, stop_r1_node, stop_r1_stop_r2, stop_r2_node, "cue_to_r1"),
        (cue_node, stop_r1_node, stop_r1_skip_r2, skip_r2_node, "cue_to_r1"),
        (cue_node, skip_r1_node, skip_r1_stop_r2, stop_r2_node, "cue_to_r1"),
        (cue_node, skip_r1_node, skip_r1_skip_r2, skip_r2_node, "cue_to_r1"),
        (stop_r1_node, stop_r2_node, stop_r1_stop_r2, stop_r2_node, "r1_to_r2"),
        (stop_r1_node, skip_r2_node, stop_r1_skip_r2, skip_r2_node, "r1_to_r2"),
        (skip_r1_node, stop_r2_node, skip_r1_stop_r2, stop_r2_node, "r1_to_r2"),
        (skip_r1_node, skip_r2_node, skip_r1_skip_r2, skip_r2_node, "r1_to_r2"),
    ]
    ordered_links = [(src, tgt, count, final_tgt, stage) for src, tgt, count, final_tgt, stage in ordered_links if count > 0]

    def link_color(src, tgt, final_tgt):
        if cue == 1:
            if tgt == stop_r1_node:
                return correct_color if final_tgt == skip_r2_node else light_green_color
            if src == stop_r1_node:
                return correct_color if tgt == skip_r2_node else light_green_color
            return double_skip_color if final_tgt == skip_r2_node else other_color
        if src == cue_node:
            if tgt == skip_r1_node:
                return correct_color if final_tgt == stop_r2_node else double_skip_color
            return light_green_color if final_tgt == stop_r2_node else other_color
        if src == skip_r1_node:
            return correct_color if tgt == stop_r2_node else double_skip_color
        return light_green_color if tgt == stop_r2_node else other_color

    sources, targets, values, colors, labels, customdata = [], [], [], [], [], []
    link_label_positions = []
    out_off = defaultdict(float)
    in_off = defaultdict(float)
    for src, tgt, count, final_tgt, stage in ordered_links:
        label = _fmt_pct(count, n_trials)
        sources.append(src)
        targets.append(tgt)
        values.append(count)
        colors.append(link_color(src, tgt, final_tgt))
        labels.append(label)
        if stage == "cue_to_r1":
            customdata.append(f"Cue {cue} -> {NODE_NAMES[tgt]} -> {NODE_NAMES[final_tgt]}")
        else:
            customdata.append(f"{NODE_NAMES[src]} -> {NODE_NAMES[tgt]}")

        seg_src_h = count / node_flow[src] * node_h[src]
        seg_tgt_h = count / node_flow[tgt] * node_h[tgt]
        y_src_center = node_y[src] + out_off[src] + seg_src_h / 2
        y_tgt_center = node_y[tgt] + in_off[tgt] + seg_tgt_h / 2
        out_off[src] += seg_src_h
        in_off[tgt] += seg_tgt_h
        label_y = (y_src_center + y_tgt_center) / 2
        if src == stop_r1_node and tgt == skip_r2_node:
            label_y -= 0.055
        elif src == skip_r1_node and tgt == stop_r2_node:
            label_y += 0.055
        if stage == "cue_to_r1":
            link_label_positions.append(dict(
                x=(node_x[src] + node_x[tgt]) / 2,
                y=label_y,
                text=label,
            ))

    trace = go.Sankey(
        arrangement="fixed",
        domain=domain,
        node=dict(
            pad=0,
            thickness=NODE_THICKNESS,
            line=dict(color="rgba(0,0,0,0.18)", width=0.4),
            label=[""] * len(node_x),
            color=node_colors,
            x=node_x,
            y=node_plot_y,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors,
            label=labels,
            customdata=customdata,
            hovertemplate="%{customdata}<br>%{label} of cue trials<extra></extra>",
        ),
    )

    summary = dict(
        cue=cue,
        n_trials=n_trials,
        node_x=node_x,
        node_y=node_y,
        node_h=node_h,
        link_label_positions=link_label_positions,
    )
    return trace, summary


def make_panel_annotations(summary, domain):
    cue = summary["cue"]
    n_trials = summary["n_trials"]
    node_x = summary["node_x"]
    node_y = summary["node_y"]
    node_h = summary["node_h"]
    x0, x1 = domain["x"]
    y0, y1 = domain["y"]
    panel_center = (x0 + x1) / 2
    panel_h = y1 - y0
    title_y = min(0.98, y1 + 0.17 * panel_h)
    header_y = min(0.95, y1 + 0.07 * panel_h)

    annotations = [
        dict(
            x=panel_center,
            y=title_y,
            xref="paper",
            yref="paper",
            text=f"<b>Cue {cue}</b> (n = {n_trials})",
            showarrow=False,
            font=dict(family="Arial", size=PANEL_TITLE_FONT_SIZE, color="#111"),
        )
    ]

    for local_x, label in [(0.50, "R1"), (0.92, "R2")]:
        paper_x, _ = _paper_point(domain, local_x, 0)
        annotations.append(dict(
            x=paper_x,
            y=header_y,
            xref="paper",
            yref="paper",
            text=f"<b>{label}</b>",
            showarrow=False,
            font=dict(family="Arial", size=10, color="#333"),
        ))

    node_text = {
        1: "Stop",
        2: "Skip",
        3: "Stop",
        4: "Skip",
    }
    for node_idx, text in node_text.items():
        center_y = node_y[node_idx] + node_h[node_idx] / 2
        paper_x, paper_y = _paper_point(domain, node_x[node_idx], center_y)
        annotations.append(dict(
            x=paper_x,
            y=paper_y,
            xref="paper",
            yref="paper",
            text=text,
            showarrow=False,
            align="center",
            font=dict(family="Arial", size=NODE_LABEL_FONT_SIZE, color="#111"),
            bgcolor="rgba(255,255,255,0.88)",
            bordercolor="rgba(0,0,0,0)",
            borderpad=2,
        ))

    for link_label in summary["link_label_positions"]:
        paper_x, paper_y = _paper_point(domain, link_label["x"], link_label["y"])
        annotations.append(dict(
            x=paper_x,
            y=paper_y,
            xref="paper",
            yref="paper",
            text=link_label["text"],
            showarrow=False,
            font=dict(family="Arial", size=FLOW_LABEL_FONT_SIZE, color="#111"),
            bgcolor="rgba(255,255,255,0.78)",
            bordercolor="rgba(0,0,0,0)",
            borderpad=1,
        ))

    return annotations


sankey_trials = prepare_choice_sankey_trials(behav)
available_sessions = sankey_trials["session_id"].unique().tolist()
if sankey_session not in available_sessions:
    raise ValueError(
        f"Session {sankey_session!r} not available. Examples: {available_sessions[:3]} ... {available_sessions[-3:]}"
    )

cue_trial_counts = (
    sankey_trials[sankey_trials["session_id"] == sankey_session]
    .groupby("cue")["trial_id"]
    .nunique()
)

sankey_fig = go.Figure()
all_annotations = []
for cue in sankey_cues_to_plot:
    domain = PANEL_DOMAINS[cue]
    trace, summary = make_cue_sankey(sankey_trials, sankey_session, cue, domain)
    sankey_fig.add_trace(trace)
    all_annotations.extend(make_panel_annotations(summary, domain))

sankey_fig.update_layout(
    annotations=all_annotations,
    showlegend=False,
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=42, r=42, t=45, b=34),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=NODE_LABEL_FONT_SIZE, color="#111"),
    hoverlabel=dict(font_family="Arial", font_size=11),
)

# Guardrails for the fixed-session figure.
for trace, cue in zip(sankey_fig.data, sankey_cues_to_plot):
    expected = int(cue_trial_counts.get(cue, 0))
    observed = sum(
        value
        for src, value in zip(trace.link.source, trace.link.value)
        if src == 0
    )
    assert observed == expected, f"Cue {cue}: first-stage link total {observed} != trial count {expected}"
    first_stage_count = sum(1 for src in trace.link.source if src == 0)
    assert first_stage_count == 4, f"Cue {cue}: expected four Cue-to-R1 branch ribbons, got {first_stage_count}"

sankey_fig.write_image(
    "figures_thesis_sam/animal_6_choice_sankey_2024-12-10_17-20.svg",
    width=sankey_fig.layout.width,
    height=sankey_fig.layout.height,
    scale=1,
)

sankey_fig


In [ ]:
# speed check
kinematics = analytics.get_analytics(
    "TrackKinematics",
    session_names=session_names,
    columns=["trial_id", "frame_velocity", "track_zone"],
)

if kinematics is None:
    raise RuntimeError("TrackKinematics could not be loaded for the selected sessions.")

expected_speed_sessions = pd.Index(
    ["_".join(session_name.split("_")[:2]) for session_name in session_names],
    dtype="object",
)
loaded_speed_sessions = pd.Index(
    kinematics.index.get_level_values("session_id").unique().astype(str),
    dtype="object",
)
missing_speed_sessions = sorted(set(expected_speed_sessions) - set(loaded_speed_sessions))

print(f"TrackKinematics loaded for {len(loaded_speed_sessions)} / {len(expected_speed_sessions)} selected sessions.")
if missing_speed_sessions:
    raise ValueError(f"Missing TrackKinematics sessions: {missing_speed_sessions}")

kinematics_speed = kinematics.reset_index()[["session_id", "trial_id", "frame_velocity", "track_zone"]].copy()
kinematics_speed["frame_velocity"] = pd.to_numeric(kinematics_speed["frame_velocity"], errors="coerce")
kinematics_speed = kinematics_speed.dropna(subset=["session_id", "trial_id", "frame_velocity"])
kinematics_speed = kinematics_speed[kinematics_speed["track_zone"].notna()]
kinematics_speed = kinematics_speed[kinematics_speed["track_zone"] != "ITI"]
kinematics_speed["speed_cm_s"] = kinematics_speed["frame_velocity"].abs()

trial_speed_summary = (
    kinematics_speed
    .groupby(["session_id", "trial_id"], observed=True)["speed_cm_s"]
    .agg(
        trial_avg_speed_cm_s="mean",
        trial_max_speed_cm_s="max",
    )
    .reset_index()
)

session_speed_summary = (
    trial_speed_summary
    .groupby("session_id", observed=True)
    .agg(
        avg_speed_cm_s=("trial_avg_speed_cm_s", "mean"),
        avg_max_speed_cm_s=("trial_max_speed_cm_s", "mean"),
        n_trials=("trial_id", "nunique"),
    )
    .reset_index()
    .sort_values("session_id")
)

missing_summary_sessions = sorted(set(expected_speed_sessions) - set(session_speed_summary["session_id"].astype(str)))
print(f"session_speed_summary contains {len(session_speed_summary)} sessions.")
if missing_summary_sessions:
    raise ValueError(f"Missing speed-summary sessions: {missing_summary_sessions}")

duplicate_speed_dates = session_speed_summary["session_id"].str.split("_").str[0].duplicated(keep=False)
speed_date_number = session_speed_summary.groupby(session_speed_summary["session_id"].str.split("_").str[0]).cumcount() + 1
session_speed_summary["session_plot_label"] = np.where(
    duplicate_speed_dates,
    session_speed_summary["session_id"].str.split("_").str[0] + " (" + speed_date_number.astype(str) + ")",
    session_speed_summary["session_id"].str.split("_").str[0],
)

# Ahmed & Mehta 2012, J Neurosci: male Long-Evans rats freely running on a Y-shaped track
# generally reached speeds up to about 100 cm/s.
# Source: https://www.researchgate.net/publication/225059336_Running_Speed_Alters_the_Frequency_of_Hippocampal_Gamma_Oscillations
long_evans_free_run_ref_cm_s = 100

# Schlachetzki et al. 2017, EJNMMI Research: sham adult male Long Evans rats
# had an average CatWalk walking speed of 46 +/- 13 cm/s.
# Source: https://link.springer.com/article/10.1186/s13550-017-0317-9
long_evans_walking_ref_cm_s = 46

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=session_speed_summary["session_plot_label"],
    y=session_speed_summary["avg_speed_cm_s"],
    mode="lines+markers",
    name="Average speed per trial",
    line=dict(width=2, color="#215CAF"),
    marker=dict(size=6, color="#215CAF"),
))
fig.add_trace(go.Scatter(
    x=session_speed_summary["session_plot_label"],
    y=session_speed_summary["avg_max_speed_cm_s"],
    mode="lines+markers",
    name="Average max speed per trial",
    line=dict(width=2, color="#B7352D"),
    marker=dict(size=6, color="#B7352D"),
))
fig.add_hline(
    y=long_evans_free_run_ref_cm_s,
    line_dash="dash",
    line_color="black",
    line_width=1,
    annotation_text="Long-Evans free-running reference (~100 cm/s)",
    annotation_position="top left",
    annotation_font=dict(size=8, family="Arial"),
)
fig.add_hline(
    y=long_evans_walking_ref_cm_s,
    line_dash="dot",
    line_color="#4F7D4A",
    line_width=1,
    annotation_text="Long-Evans walking reference (~46 cm/s)",
    annotation_position="bottom left",
    annotation_font=dict(size=8, family="Arial"),
)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=session_speed_summary["session_plot_label"],
    tickangle=-45,
    tickfont=dict(size=7, family="Arial"),
    title_font=dict(size=8, family="Arial"),
    title_standoff=4,
    showgrid=False,
)
fig.update_yaxes(
    title_text="Speed (cm/s)",
    tickfont=dict(size=8, family="Arial"),
    title_font=dict(size=9, family="Arial"),
    showgrid=True,
    gridcolor="rgba(128,128,128,0.35)",
    gridwidth=0.5,
    zeroline=False,
)
fig.update_layout(
    title=dict(text="Session speed summary", font=dict(size=10, family="Arial")),
    width=900,
    height=340,
    margin=dict(l=55, r=30, t=35, b=95),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=9),
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor="white",
        bordercolor="rgba(0,0,0,0.2)",
        borderwidth=1,
        xanchor="left",
        yanchor="top",
        font=dict(size=9, family="Arial"),
    ),
)

fig.show()

session_speed_summary